# Fish detector fine-tuned on DeepFish - Colab, ~1 h on a T4

**Why:** the Brackish YOLOv8s model (99.3% mAP50 on its own data) scored only **0.5% AP50** on the DeepFish
test split (`outputs/metrics/deepfish_brackish_yolov8s_test.json`). Here we train on DeepFish itself and test on
**the same test split with the same rules**, so the two numbers are directly comparable.

**Data:** DeepFish (Saleh et al. 2020, James Cook University) with fish boxes released by the YOLO-Fish authors
(github.com/tamim662/YOLO-Fish): the full annotated set and their separate 909-frame test set.

| Part | What | Used for |
|---|---|---|
| train | full annotated set, **minus every frame from a clip that appears in the 909-frame test set** | training |
| train-val | 10% of the remaining training clips | picking the best epoch |
| eval-val | 30% of the test-set clips (same md5 split as the local script) | picking the confidence threshold only |
| eval-test | the other 70% (758 frames, 35 clips, 2,767 fish) | **reported once** |

Matching: greedy, highest confidence first, IoU >= 0.5. AP50 + P/R/F1, 95% bootstrap CIs over test clips.

**Runtime:** `Runtime -> Change runtime type -> T4 GPU`, then `Runtime -> Run all`.
**Bring back:** `MyDrive/fish/deepfish_finetune_results.json` (and `best_deepfish.pt` if you want to use the model).

In [ ]:
!nvidia-smi
!pip install -q -U ultralytics gdown

In [ ]:
MODEL = 'yolo11s.pt'   # COCO-pretrained start; yolo11m.pt is stronger but ~2x slower
IMGSZ = 768
BATCH = 16             # lower to 8 if CUDA runs out of memory
EPOCHS = 50
PATIENCE = 15
SEED = 42
RUN_NAME = 'yolo11s_deepfish'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, re, json, hashlib, shutil, random, collections, zipfile
import numpy as np
from pathlib import Path
from PIL import Image
DRIVE_DIR = '/content/drive/MyDrive/fish'; os.makedirs(DRIVE_DIR, exist_ok=True)
ROOT = Path('/content/deepfish'); ROOT.mkdir(exist_ok=True)

# Google Drive ids from github.com/tamim662/YOLO-Fish
FILES = {'full.zip': '10Pr4lLeSGTfkjA40ReGSC8H3a9onfMZ0', 'test.zip': '1iHPFbqo-B2iVytusHic9s8VuMlLIMu5-'}
for name, fid in FILES.items():
    if not (ROOT / name).exists():
        !gdown {fid} -O {str(ROOT / name)}
    out = ROOT / name.replace('.zip', '')
    if not out.exists():
        with zipfile.ZipFile(ROOT / name) as z:
            z.extractall(out)

def labelled_images(folder):
    # every .jpg/.png that has a YOLO .txt label next to it
    return sorted(p for p in Path(folder).rglob('*') if p.suffix.lower() in ('.jpg', '.jpeg', '.png')
                  and p.with_suffix('.txt').exists())

FULL = labelled_images(ROOT / 'full'); TEST = labelled_images(ROOT / 'test')
print('full annotated set:', len(FULL), 'labelled images | test set:', len(TEST), 'labelled images')
print('example names:', [p.name for p in FULL[:3]])
assert len(TEST) == 909, 'test set should have 909 frames'

## Leak-free split
Frames from the same video clip are near-duplicates, so any training frame whose clip is in the test set is removed
(not just identical file names). The clip / eval split functions are copied from
`src/evaluation/deepfish_brackish_eval.py`, so eval-val and eval-test are exactly the frames used for the Brackish run.

In [ ]:
def clip(n):  return re.sub(r'_f\d+$', '', n)
def split(n): return 'val' if int(hashlib.md5(clip(n).encode()).hexdigest(), 16) % 10 < 3 else 'test'

test_clips = {clip(p.stem) for p in TEST}
test_names = {p.stem for p in TEST}
pool = [p for p in FULL if clip(p.stem) not in test_clips]
print('training pool after removing test clips:', len(pool), 'of', len(FULL),
      f'({len(FULL) - len(pool)} removed, {sum(p.stem in test_names for p in FULL)} were identical file names)')
print('clips: full', len({clip(p.stem) for p in FULL}), '| test', len(test_clips), '| training pool', len({clip(p.stem) for p in pool}))
assert not ({clip(p.stem) for p in pool} & test_clips)
assert len(pool) > 300, 'too few training frames left - check the clip naming in the printout above'

# train / train-val split by clip (for best-epoch selection only)
pool_clips = sorted({clip(p.stem) for p in pool})
random.Random(SEED).shuffle(pool_clips)
tv_clips = set(pool_clips[:max(1, len(pool_clips) // 10)])
DS = Path('/content/deepfish_yolo')
if DS.exists(): shutil.rmtree(DS)
for p in pool:
    s = 'val' if clip(p.stem) in tv_clips else 'train'
    for src, sub in ((p, 'images'), (p.with_suffix('.txt'), 'labels')):
        d = DS / s / sub; d.mkdir(parents=True, exist_ok=True)
        shutil.copy(src, d / src.name)
# DeepFish-YOLO-Fish labels: single class 0 = fish; force class 0 in case of stray ids
for f in DS.rglob('labels/*.txt'):
    rows = [l.split() for l in f.read_text().splitlines() if l.strip()]
    f.write_text(''.join('0 ' + ' '.join(r[1:5]) + '\n' for r in rows if len(r) >= 5))
(DS / 'data.yaml').write_text(f'path: {DS}\ntrain: train/images\nval: val/images\nnames:\n  0: fish\n')
for s in ('train', 'val'):
    print(s, len(list((DS / s / 'images').iterdir())), 'images')

In [ ]:
from ultralytics import YOLO
model = YOLO(MODEL)
model.train(data=str(DS / 'data.yaml'), imgsz=IMGSZ, epochs=EPOCHS, batch=BATCH, patience=PATIENCE, seed=SEED,
            project='/content/runs', name=RUN_NAME, exist_ok=True, plots=True)
BEST = f'/content/runs/{RUN_NAME}/weights/best.pt'
shutil.copy(BEST, f'{DRIVE_DIR}/best_deepfish.pt')
print('saved', f'{DRIVE_DIR}/best_deepfish.pt')

## Evaluation on the DeepFish test split (same code as the Brackish run)

In [ ]:
def load_gt(img):
    w, h = Image.open(img).size
    rows = [list(map(float, l.split()[1:5])) for l in img.with_suffix('.txt').read_text().splitlines() if l.strip()]
    b = np.array(rows, np.float32).reshape(-1, 4)
    return np.stack([(b[:, 0] - b[:, 2] / 2) * w, (b[:, 1] - b[:, 3] / 2) * h,
                     (b[:, 0] + b[:, 2] / 2) * w, (b[:, 1] + b[:, 3] / 2) * h], 1)

def iou(a, b):
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area = lambda z: (z[:, 2] - z[:, 0]) * (z[:, 3] - z[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-9)

def match_flags(gt, boxes, confs):
    order = np.argsort(-confs)
    ious = iou(boxes, gt) if len(boxes) and len(gt) else np.zeros((len(boxes), len(gt)))
    used, out = set(), []
    for i in order:
        cand = [(ious[i, g], g) for g in range(len(gt)) if g not in used and ious[i, g] >= 0.5]
        if cand:
            used.add(max(cand)[1])
        out.append((float(confs[i]), bool(cand)))
    return out

def counts(gt, boxes, confs, conf):
    k = confs >= conf
    f = match_flags(gt, boxes[k], confs[k]); tp = sum(x for _, x in f)
    return tp, len(f) - tp, len(gt) - tp

def prf(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else 0.0; r = tp / (tp + fn) if tp + fn else 0.0
    return p, r, (2 * p * r / (p + r) if p + r else 0.0)

def ap50(gt, preds, ns):
    flags = sorted((f for n in ns for f in match_flags(gt[n], *preds[n])), key=lambda t: -t[0])
    n_gt = sum(len(gt[n]) for n in ns)
    tps = np.cumsum([f for _, f in flags]); fps = np.cumsum([not f for _, f in flags])
    mrec = np.concatenate([[0], tps / n_gt, [1]]); mpre = np.concatenate([[1], tps / np.maximum(tps + fps, 1), [0]])
    mpre = np.flip(np.maximum.accumulate(np.flip(mpre)))
    return float(np.sum((mrec[1:] - mrec[:-1]) * mpre[1:]))

def evaluate(gt, preds, splits):
    tot = lambda ns, c: tuple(sum(v) for v in zip(*(counts(gt[n], *preds[n], c) for n in ns)))
    table = [(prf(*tot(splits['val'], c))[2], float(c)) for c in np.round(np.arange(0.05, 0.91, 0.05), 2)]
    vf1, conf = max(table)
    ns = splits['test']
    per = {n: counts(gt[n], *preds[n], conf) for n in ns}
    tp, fp, fn = (sum(v[k] for v in per.values()) for k in range(3)); p, r, f1 = prf(tp, fp, fn)
    groups = collections.defaultdict(list)
    for n in ns:
        groups[clip(n)].append(n)
    rng = np.random.default_rng(0); keys = list(groups); boot = []
    for _ in range(1000):
        s = [n for k in rng.choice(keys, len(keys)) for n in groups[k]]
        boot.append(prf(*(sum(per[n][k] for n in s) for k in range(3))))
    ci = lambda j: [round(float(np.percentile([b[j] for b in boot], q)), 3) for q in (2.5, 97.5)]
    return {'conf_chosen_on_val': conf, 'val_f1': round(vf1, 3), 'precision': round(p, 3), 'recall': round(r, 3),
            'f1': round(f1, 3), 'ap50': round(ap50(gt, preds, ns), 3), 'tp': int(tp), 'fp': int(fp), 'fn': int(fn),
            'precision_ci': ci(0), 'recall_ci': ci(1), 'f1_ci': ci(2)}

GT = {p.stem: load_gt(p) for p in TEST}
SPLITS = {s: sorted(n for n in GT if split(n) == s) for s in ('val', 'test')}
for s, ns in SPLITS.items():
    print(s, len(ns), 'frames', len({clip(n) for n in ns}), 'clips', sum(len(GT[n]) for n in ns), 'fish')
assert len(SPLITS['test']) == 758, 'eval-test must be the same 758 frames as the Brackish run'

m = YOLO(BEST); PREDS = {}
for i in range(0, len(TEST), 16):
    batch = TEST[i:i + 16]
    for p, r in zip(batch, m.predict([str(x) for x in batch], conf=0.02, imgsz=IMGSZ, device=0, verbose=False)):
        PREDS[p.stem] = (r.boxes.xyxy.cpu().numpy().astype(np.float32), r.boxes.conf.cpu().numpy().astype(np.float32))

res = evaluate(GT, PREDS, SPLITS)
RESULTS = {
    'source': 'notebooks/colab_deepfish_finetune.ipynb (Colab)',
    'model': f'{MODEL} fine-tuned on DeepFish (YOLO-Fish boxes), imgsz {IMGSZ}, {EPOCHS} epochs max, patience {PATIENCE}',
    'train_images': len(list((DS / 'train' / 'images').iterdir())), 'train_val_images': len(list((DS / 'val' / 'images').iterdir())),
    'leak_check': 'every frame from a clip present in the 909-frame test set removed from training',
    'dataset': 'DeepFish test set (YOLO-Fish boxes, 909 frames); split by clip 30% val (threshold only) / 70% test',
    'test_frames': len(SPLITS['test']), 'test_clips': len({clip(n) for n in SPLITS['test']}),
    'test_fish': int(sum(len(GT[n]) for n in SPLITS['test'])), 'matching': 'greedy IoU>=0.5',
    'test': res,
    'baseline_same_split': {'model': 'dronefreak/brackish-yolov8s, no fine-tuning', 'ap50': 0.005,
                            'precision': 0.047, 'recall': 0.023, 'f1': 0.031},
}
json.dump(RESULTS, open(f'{DRIVE_DIR}/deepfish_finetune_results.json', 'w'), indent=1)
print(json.dumps(RESULTS, indent=1))

## A few test frames (green = label, model boxes drawn by Ultralytics)

In [ ]:
import cv2, matplotlib.pyplot as plt
picks = random.Random(0).sample(SPLITS['test'], 4)
fig, axes = plt.subplots(2, 2, figsize=(16, 9))
for ax, n in zip(axes.flat, picks):
    p = next(x for x in TEST if x.stem == n)
    r = m.predict(str(p), conf=res['conf_chosen_on_val'], imgsz=IMGSZ, verbose=False)[0]
    im = r.plot()
    for b in GT[n].astype(int):
        cv2.rectangle(im, (b[0], b[1]), (b[2], b[3]), (0, 255, 0), 3)
    ax.imshow(im[:, :, ::-1]); ax.set_title(n); ax.axis('off')
plt.tight_layout(); plt.savefig(f'{DRIVE_DIR}/deepfish_finetune_examples.png', dpi=100); plt.show()